# Week-08 · Chapter 07 — Deep Learning & Generative AI

이번 Notebook은 **독립적으로 실행할 수 있는 Colab 실습 자료**입니다. 수업 본문은 `README.md`를 참고하세요.

- 7.1: NumPy 기반 순전파 계산
- 7.2: PyTorch 기반 Digits(8×8) 숫자 분류 학습
- 7.3: MobileNetV3 사전 학습 이미지 분류
- 7.4: SmolLM2 소형 언어 모델 텍스트 생성

**7.3·7.4는 최초 실행 시 인터넷으로 모델 가중치를 다운로드합니다.** GPU가 없어도 7.1·7.2는 CPU로 실습할 수 있습니다.

## 실행 환경 준비

기본 라이브러리를 설치합니다. Colab에 이미 설치되어 있다면 재설치할 필요는 없습니다. GPU 런타임을 선택해도 CUDA 지원 빌드가 없으면 CPU로 동작할 수 있습니다.

In [ ]:
%pip install -q numpy matplotlib scikit-learn pillow transformers

In [ ]:
import torch
import sklearn
import transformers
try:
    import torchvision
    tv = torchvision.__version__
except ImportError:
    tv = "설치 필요 (7.3에서 사용)"

print("PyTorch:", torch.__version__)
print("TorchVision:", tv)
print("scikit-learn:", sklearn.__version__)
print("Transformers:", transformers.__version__)
print("GPU 사용 가능:", torch.cuda.is_available())

> `torch` 또는 `torchvision`이 없다면 별도 셀에서 `%pip install torch torchvision`을 실행하세요. PyTorch와 TorchVision은 호환되는 버전으로 설치해야 합니다. 패키지 설치 중 런타임 재시작 안내가 나오면 재시작 후 처음부터 실행하세요.

## 7.1. Introduction to Deep Learning

**입력 2개 → 은닉 뉴런 3개 → 출력 1개** 구조를 계산합니다. 아래 코드는 모델 학습이 아닌 **순전파** 예제입니다.

- `x`: 입력값 2개
- `W1`, `b1`: 입력에서 은닉층으로 가는 가중치·편향
- `z = x @ W1 + b1`: 은닉층 가중합 (`@`는 행렬 곱셈)
- `hidden = np.maximum(z, 0)`: ReLU. **-0.06 → 0** 변화 확인
- `W2`, `b2`: 은닉층에서 최종 출력으로 가는 가중치·편향
- 예상 출력: **0.782**. 지정한 값으로 계산한 숫자이며 학습된 예측이나 확률이 아닙니다.

자세한 변수별 표와 실제 계산은 `README.md`의 **7.1.3**을 참조하세요.

### 7.1.3. Activation Functions — Sigmoid / ReLU / Softmax / GELU

- **Sigmoid**: 하나의 값을 0과 1 사이로 변환합니다. 이진 분류의 출력 해석에 사용합니다. `σ(x) = 1/(1+e^(-x))`
- **ReLU**: 음수는 0, 양수는 그대로 전달합니다. 은닉층에서 널리 사용합니다. `ReLU(x) = max(0, x)`
- **Softmax**: 여러 클래스의 점수를 합이 1인 값으로 변환합니다. 다중 분류 결과를 해석할 때 사용합니다.
- **GELU**: `x·Φ(x)`로 정의되며, ReLU보다 부드럽게 값을 변환합니다. 일부 Transformer 모델에서 사용합니다.

**주의:** 7.2의 `CrossEntropyLoss()`는 Logits를 직접 입력받으므로 학습 코드에서 Softmax를 미리 적용하지 않습니다. 아래 코드는 함수별 변화와 Softmax 출력 분포를 직접 관찰하는 체험 실습입니다. 결과 그래프는 `assets/`에 저장됩니다.


In [ ]:
"""7.1 — Sigmoid, ReLU, Softmax, GELU의 출력과 그래프 비교.

Google Colab 또는 CPU Python에서 실행할 수 있습니다.
필요 라이브러리: numpy, torch, matplotlib
"""
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.nn import functional as F

# 노트북에서는 현재 폴더, Python 파일에서는 Week-08 폴더에 이미지 저장
OUT_DIR = (
    Path(__file__).resolve().parents[1] / "assets"
    if "__file__" in globals()
    else Path("assets")
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. 단일 입력에 적용하는 활성화 함수 비교
samples = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
print("입력값: ", samples.tolist())
for name, result in (
    ("Sigmoid", torch.sigmoid(samples)),
    ("ReLU", F.relu(samples)),
    ("GELU", F.gelu(samples)),
):
    print(f"{name:>8}: ", [round(float(v), 4) for v in result])

# 2. 곡선 시각화
x = torch.linspace(-4, 4, 400)
fig, ax = plt.subplots(figsize=(9, 5.4), dpi=175)
ax.plot(x.numpy(), torch.sigmoid(x).numpy(), label="Sigmoid", linewidth=2.5)
ax.plot(x.numpy(), F.relu(x).numpy(), label="ReLU", linewidth=2.5)
ax.plot(x.numpy(), F.gelu(x).numpy(), label="GELU", linewidth=2.5)
ax.axhline(0, linewidth=0.7, alpha=0.4, color="gray")
ax.axvline(0, linewidth=0.7, alpha=0.4, color="gray")
ax.set_title("Activation Functions: Sigmoid, ReLU, GELU", fontsize=16, pad=14)
ax.set_xlabel("Input (x)")
ax.set_ylabel("Output")
ax.set_xlim(-4, 4)
ax.set_ylim(-0.6, 4.25)
ax.grid(alpha=0.23)
ax.legend(loc="upper left")
fig.tight_layout()
fig.savefig(OUT_DIR / "7_1_activation_curves.png", dpi=175, facecolor="white")
plt.show()
plt.close(fig)

# 3. Softmax는 여러 클래스의 점수를 함께 변환함
logits = torch.tensor([1.0, 2.0, 3.0])  # A, B, C에 대한 원시 점수
probs = F.softmax(logits, dim=0)
print("Softmax 입력(원시 점수):", logits.tolist())
print("Softmax 출력(분포):", [round(float(v), 4) for v in probs])
print("출력값의 합:", round(float(probs.sum()), 4))

labels = ["Class A", "Class B", "Class C"]
fig, ax = plt.subplots(figsize=(8.6, 5.2), dpi=175)
bars = ax.bar(labels, (probs * 100).numpy(), width=0.55)
for bar, p in zip(bars, probs):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1.5, f"{float(p):.1%}",
            ha="center", fontsize=12)
ax.set_ylim(0, 80)
ax.set_ylabel("Softmax Output (%)")
ax.set_title("Softmax: Scores [1, 2, 3] → Class Distribution", fontsize=15, pad=14)
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(OUT_DIR / "7_1_softmax_example.png", dpi=175, facecolor="white")
plt.show()
plt.close(fig)


### 7.1.4. Coding Practice — 신경망의 순전파
정해진 가중치를 사용하여 입력 2개 → 은닉 뉴런 3개 → 최종 출력 1개를 계산합니다. ReLU가 `-0.06`을 `0`으로 변경하는 부분을 확인하세요.


In [ ]:
"""7.1 — 입력 2개 → 은닉 뉴런 3개 → 출력 1개의 순전파 계산.

NumPy와 고정된 가중치로 계산만 확인하는 예제입니다. 모델 학습은 수행하지 않습니다.
"""
import numpy as np

# 1. 입력값 두 개 (임의의 예시 값이며 실제 관측값이 아님)
x = np.array([0.6, 0.8])

# 2. 첫 번째 연결: 입력 2개 → 은닉 뉴런 3개
# W1: (2, 3) 형태의 가중치, b1: 은닉 뉴런마다 더할 편향 3개
# 두 번째 뉴런의 가중치를 음수로 설정해 ReLU의 변화를 확인합니다.
W1 = np.array([[0.4, -0.9, 0.7],
               [0.2,  0.6, -0.1]])
b1 = np.array([0.1, 0.0, 0.2])

# 3. 두 번째 연결: 은닉 뉴런 3개 → 최종 출력 1개
W2 = np.array([[0.5], [-0.4], [0.8]])
b2 = np.array([0.1])

# 4. 순전파: 가중합 → ReLU → 최종 출력
z = x @ W1 + b1            # @: 입력과 가중치의 행렬 곱셈
hidden = np.maximum(z, 0)   # ReLU: 음수는 0, 0 이상은 그대로
output = hidden @ W2 + b2  # 출력층에서는 ReLU를 사용하지 않음

print("입력:", x)
print("은닉층 가중합:", np.round(z, 3))
print("은닉층 ReLU 출력:", np.round(hidden, 3))
print("최종 출력:", np.round(output, 3))
print("참고: 고정된 가중치로 계산만 하며 학습하는 모델은 아닙니다.")


## 7.2. Neural Networks & Model Training

`scikit-learn`의 **Digits 데이터셋(8×8 이미지, 1,797개)**을 이용하여 숫자 0~9를 분류합니다.

1. `digits.data / 16.0`: 픽셀의 0~16 값을 0~1로 정규화
2. `DataLoader`: 입력·정답을 **Batch 64개**씩 읽음
3. `nn.Linear(64, HIDDEN_UNITS) → ReLU → nn.Linear(HIDDEN_UNITS, 10)`: 신경망
4. `zero_grad → model → loss → backward → step`: 한 Batch의 학습 순서
5. 최종 **Test Accuracy**와 **Epoch별 Train Loss** 및 숫자 예측 이미지 확인

`EPOCHS`(학습 횟수)와 `HIDDEN_UNITS`(은닉 노드 수)를 바꿔 봅니다. 후보 설정 비교는 마지막 Exercise에서 **Validation** 데이터로 수행합니다.
**PyTorch의 기본 구조**: `torch`는 텐서 연산 및 자동 미분을 위한 라이브러리, `nn`은 신경망의 층과 손실 함수를 위한 모듈입니다. `nn.Linear(in_features, out_features)`의 두 숫자는 **입력 특징 수**와 **출력값 수**입니다. `nn.Sequential()`은 여러 층을 순서대로 연결하고, `nn.ReLU()`는 비선형 변환을 적용합니다. **8×8 픽셀 64개 → 은닉 뉴런 64개 → 숫자 점수 10개** 구조를 기억하세요.

신경망 층별 입력과 출력 구조는 README의 **그림 7.2 — Linear Layer 구조**를 참조하세요.

모델 학습에서는 `model.train()`, 평가에서는 `model.eval()`을 사용합니다. `model.eval()`은 기울기 계산을 끄지 않으므로 `torch.no_grad()` 또는 `torch.inference_mode()`를 별도로 사용합니다. EPOCHS와 HIDDEN_UNITS는 직접 바꿀 수 있는 **하이퍼파라미터**이며, Exercise 01에서는 Validation 정확도로 비교합니다.


### 7.2.2. 손실 함수 — MSE / BCE / Cross-Entropy

- 회귀: **MSE** — 실제값과 예측값의 차이를 제곱해 평균 계산
- 이진 분류: **BCE** — Sigmoid 출력과 이진 정답의 관계 평가. `BCEWithLogitsLoss`는 **Logit**을 직접 입력받음
- 다중 클래스 분류: **Cross-Entropy** — 정답 클래스에 부여한 확률을 평가. `CrossEntropyLoss`는 **Logit**을 직접 입력받음

**학습 시** `BCEWithLogitsLoss` 앞에 Sigmoid를, `CrossEntropyLoss` 앞에 Softmax를 따로 적용하지 않습니다. 출력 확률을 확인하는 작업과 손실을 계산하는 작업을 구분합니다.

수식과 예제 설명은 `README.md`의 7.2.2를 참고하세요.


In [ ]:
"""7.2 — MSE, BCEWithLogitsLoss, CrossEntropyLoss 비교.

간단한 가상 데이터를 사용하여 올바른 예측과 잘못된 예측의 손실을 계산합니다.
CPU에서 실행 가능하며, assets/7_2_loss_comparison.png를 생성합니다.
"""
from pathlib import Path
import matplotlib.pyplot as plt
import torch
from torch import nn

OUT_DIR = (Path(__file__).resolve().parents[1] / "assets") if "__file__" in globals() else Path("assets")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. 회귀: 정답과 예측값 간 제곱 오차의 평균
mse_fn = nn.MSELoss()
actual_scores = torch.tensor([80.0, 90.0])
good_scores = torch.tensor([78.0, 91.0])
bad_scores = torch.tensor([65.0, 98.0])
mse_good = mse_fn(good_scores, actual_scores).item()
mse_bad = mse_fn(bad_scores, actual_scores).item()

# 2. 이진 분류: 정답이 1일 때, Sigmoid 전 원시 점수(Logit)로 손실 계산
bce_fn = nn.BCEWithLogitsLoss()
correct_binary = torch.tensor([1.0])     # 실수형 0 또는 1
logit_good = torch.tensor([2.0])         # Sigmoid 결과 약 0.881
logit_bad = torch.tensor([-2.0])         # Sigmoid 결과 약 0.119
bce_good = bce_fn(logit_good, correct_binary).item()
bce_bad = bce_fn(logit_bad, correct_binary).item()

# 3. 다중 클래스 분류: 클래스 0, 1, 2 중 정답이 2
ce_fn = nn.CrossEntropyLoss()
correct_class = torch.tensor([2])        # 정답 클래스 인덱스(정수)
logits_good = torch.tensor([[1.0, 2.0, 3.0]])
logits_bad = torch.tensor([[3.0, 2.0, 1.0]])
ce_good = ce_fn(logits_good, correct_class).item()
ce_bad = ce_fn(logits_bad, correct_class).item()

print(f"MSE  | 좋은 예측: {mse_good:.3f} | 나쁜 예측: {mse_bad:.3f}")
print(f"BCE  | 좋은 예측: {bce_good:.3f} | 나쁜 예측: {bce_bad:.3f}")
print(f"CE   | 좋은 예측: {ce_good:.3f} | 나쁜 예측: {ce_bad:.3f}")
print("BCE 확률 비교:", round(torch.sigmoid(logit_good).item(), 3),
      round(torch.sigmoid(logit_bad).item(), 3))
print("CE 정답 클래스 확률:",
      round(torch.softmax(logits_good, dim=1)[0, 2].item(), 3),
      round(torch.softmax(logits_bad, dim=1)[0, 2].item(), 3))

# 각 손실의 수치 범위가 다르므로 개별 y축으로 그립니다.
metrics = [
    ("MSE · Regression", mse_good, mse_bad),
    ("BCE · Binary Classification", bce_good, bce_bad),
    ("CE · Multiclass Classification", ce_good, ce_bad),
]
fig, axes = plt.subplots(1, 3, figsize=(12.6, 4.3), dpi=180)
for ax, (title, good, bad) in zip(axes, metrics):
    bars = ax.bar(["Good", "Incorrect"], [good, bad], width=0.52,
                  color=["#508CA2", "#CB8F65"])
    ax.set_title(title, fontsize=11, pad=14)
    ax.set_ylabel("Loss (lower is better)")
    ax.set_ylim(0, max(good, bad)*1.24)
    ax.grid(axis="y", alpha=0.18)
    ax.set_axisbelow(True)
    for b, val in zip(bars, [good, bad]):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+max(good, bad)*0.05,
                f"{val:.3f}", fontsize=11, ha="center")
fig.suptitle("Loss Functions — Correct vs. Incorrect Predictions", fontsize=15, y=1.05)
fig.text(0.5, -0.02, "Different loss scales: do not compare heights across tasks.",
         fontsize=10, color="#666666", ha="center")
fig.tight_layout()
fig.savefig(OUT_DIR / "7_2_loss_comparison.png", dpi=180, facecolor="white", bbox_inches="tight")
plt.show()
plt.close(fig)


In [ ]:
"""7.2 — PyTorch로 scikit-learn Digits(8×8) 분류 학습.

별도 데이터 파일 불필요. CPU 실행 가능. 실행 후 이미지 두 개를 assets/에 저장합니다.
"""
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
EPOCHS = 15            # 학습 반복 횟수: 5, 15, 30으로 바꾸어 보세요.
HIDDEN_UNITS = 64       # 은닉층 노드: 16, 64, 128을 비교해 보세요.
BATCH_SIZE = 64
LEARNING_RATE = 0.01

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(2, torch.get_num_threads()))

# Notebook에서는 assets/, Python 파일 실행 시에는 Week-08/assets/ 사용
out_dir = (Path(__file__).resolve().parents[1] / "assets") if "__file__" in globals() else Path("assets")
out_dir.mkdir(parents=True, exist_ok=True)

digits = load_digits()
X = (digits.data / 16.0).astype(np.float32)  # 8x8 → 특징 64개, 픽셀값 0~1
y = digits.target.astype(np.int64)          # 정답: 0~9
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
    batch_size=BATCH_SIZE, shuffle=True
)

model = nn.Sequential(
    nn.Linear(64, HIDDEN_UNITS),
    nn.ReLU(),
    nn.Linear(HIDDEN_UNITS, 10),
)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

loss_history = []
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for features, labels in train_loader:
        optimizer.zero_grad()
        logits = model(features)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
    train_loss = total_loss / len(X_train)
    loss_history.append(train_loss)
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train Loss: {train_loss:.4f}")

model.eval()
with torch.inference_mode():
    predictions = model(torch.from_numpy(X_test)).argmax(dim=1).numpy()
accuracy = float(np.mean(predictions == y_test))
print(f"Test Accuracy: {accuracy:.1%} ({np.sum(predictions == y_test)}/{len(y_test)})")

# Figure 1 — 학습 손실 곡선
fig, ax = plt.subplots(figsize=(9.6, 5.4), dpi=170)
ax.plot(range(1, EPOCHS+1), loss_history, marker="o", linewidth=2.4, markersize=5)
ax.set_title("Training Loss by Epoch", fontsize=16, pad=16)
ax.set_xlabel("Epoch")
ax.set_ylabel("Train Loss")
ax.set_xticks(range(1, EPOCHS+1))
ax.grid(alpha=0.22)
fig.tight_layout()
fig.savefig(out_dir / "7_2_training_curve.png", facecolor="white", dpi=170)
plt.show()

# Figure 2 — 테스트 이미지와 예측 숫자 비교
fig, axes = plt.subplots(3, 4, figsize=(10, 8.1), dpi=170)
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test[i].reshape(8, 8), cmap="gray_r", interpolation="nearest")
    correct = predictions[i] == y_test[i]
    ax.set_title(f"True {y_test[i]}  /  Pred {predictions[i]}" + ("" if correct else "  *"), fontsize=10)
    ax.axis("off")
fig.suptitle("Handwritten Digit Classification — Test Samples", fontsize=15)
fig.tight_layout(pad=1.9, h_pad=2.2)
fig.savefig(out_dir / "7_2_digits_predictions.png", facecolor="white", dpi=170)
plt.show()


## 7.3. Computer Vision & Image Recognition

**사전 학습된 MobileNetV3-Small**로 사진을 분류합니다. 처음 실행할 때 모델 가중치와 예제 사진을 다운로드합니다.

- `weights.transforms()`: 이미지 크기·값을 모델 입력에 맞게 전처리
- `unsqueeze(0)`: 사진 한 장을 Batch 형태 `(1, 3, 224, 224)`로 변환
- `model.eval()` / `torch.inference_mode()`: **학습 없이 추론만 수행**
- Softmax 점수 상위 5개를 출력하며 `assets/7_3_image_prediction.png`로 저장

`UPLOAD_OWN_IMAGE=True`로 바꾸면 Colab에서 직접 이미지 파일을 업로드할 수 있습니다. 실제 추론 결과는 가중치를 내려받아 실행한 후 확인할 수 있습니다.
`UPLOAD_OWN_IMAGE = False`는 기본 공개 사진, `True`는 Colab 업로드 사진을 사용한다는 뜻입니다. 7.3에서는 모델을 재학습하지 않고 입력 사진만 바꿔 이미 학습된 모델의 추론 결과를 비교합니다.


In [ ]:
"""7.3 — MobileNetV3-Small 사전 학습 모델을 이용한 이미지 분류.

첫 실행 시 공개 샘플 사진과 pretrained 가중치를 내려받으므로 인터넷 연결이 필요합니다.
UPLOAD_OWN_IMAGE=True로 바꾸면 Colab에서 직접 이미지 파일을 선택할 수 있습니다.
"""
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import torch
from PIL import Image
from torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small

UPLOAD_OWN_IMAGE = False
SAMPLE_IMAGE_URL = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"

out_dir = (Path(__file__).resolve().parents[1] / "assets") if "__file__" in globals() else Path("assets")
out_dir.mkdir(parents=True, exist_ok=True)

if UPLOAD_OWN_IMAGE:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("사진 업로드는 Google Colab에서 실행하거나 코드를 로컬 경로로 수정하세요.") from exc
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("분류할 이미지를 업로드하세요.")
    img_path = Path(next(iter(uploaded)))
else:
    img_path = out_dir / "sample_dog.jpg"
    if not img_path.is_file():
        print("공개 예제 사진 다운로드 중...")
        urlretrieve(SAMPLE_IMAGE_URL, img_path)

image = Image.open(img_path).convert("RGB")
weights = MobileNet_V3_Small_Weights.DEFAULT
model = mobilenet_v3_small(weights=weights)
model.eval()

input_tensor = weights.transforms()(image).unsqueeze(0)
with torch.inference_mode():
    logits = model(input_tensor)
    scores = logits.softmax(dim=1)[0]
    top_scores, top_ids = torch.topk(scores, k=5)

classes = weights.meta["categories"]
names = [classes[i] for i in top_ids.tolist()]
values = top_scores.tolist()

print("ImageNet Top-5 Prediction")
for rank, (name, score) in enumerate(zip(names, values), 1):
    print(f"{rank}. {name}: {score:.1%}")

fig, (ax_image, ax_score) = plt.subplots(1, 2, figsize=(12.2, 5), dpi=170,
                                        gridspec_kw={"width_ratios": [1, 1.3]})
ax_image.imshow(image)
ax_image.set_title("Input Image")
ax_image.axis("off")
ax_score.barh(names[::-1], [v * 100 for v in values[::-1]])
ax_score.set_title("Top 5 Predictions")
ax_score.set_xlabel("Softmax Score (%)")
ax_score.set_xlim(0, 100)
ax_score.grid(axis="x", alpha=0.18)
ax_score.set_axisbelow(True)
fig.tight_layout()
result = out_dir / "7_3_image_prediction.png"
fig.savefig(result, facecolor="white", dpi=170, bbox_inches="tight")
plt.show()
print("그래프 저장:", result)
print("참고: Softmax 출력은 실제 정답일 확률을 보장하지 않습니다.")


## 7.4. Generative AI & Large Language Models

**SmolLM2-135M-Instruct**로 짧은 영어 텍스트를 생성합니다. 처음 실행할 때 공개 모델을 다운로드합니다.

- `PROMPT`: 모델에 전달할 문장
- `apply_chat_template()`: 대화 메시지를 토큰 ID로 변환
- `max_new_tokens`: 생성할 새 토큰의 최대 개수 (글자 수가 아님)
- `generated_ids[0][input_ids.shape[-1]:]`: 입력 부분을 빼고 새로 생성한 토큰만 선택
- `tokenizer.decode()`: 토큰 ID를 읽을 수 있는 문자로 변환

작은 모델은 명령을 잘 지키지 못하거나 부정확한 답을 할 수 있습니다. 실제 생성 결과는 Colab에서 확인하세요.
`max_new_tokens`는 글자 수가 아니라 **새 토큰을 생성할 수 있는 최대 개수**입니다. 30과 70을 비교하더라도 항상 제한까지 생성되는 것은 아닙니다. 생성 문장의 사실성은 별도로 검증해야 합니다.


In [ ]:
"""7.4 — 공개 소형 언어 모델(SmolLM2-135M-Instruct)을 이용한 텍스트 생성.

처음 실행할 때 Hugging Face에서 가중치를 다운로드하므로 인터넷이 필요합니다.
주로 영어로 학습된 소형 모델이므로 영어 프롬프트를 권장합니다.
"""
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
PROMPT = "List three simple ways to save electricity at home. Keep each tip short."
MAX_NEW_TOKENS = 70

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("첫 실행 시 공개 모델을 다운로드합니다.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(device)
model.eval()

messages = [{"role": "user", "content": PROMPT}]
input_ids = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True,
    return_tensors="pt"
).to(device)

with torch.inference_mode():
    generated_ids = model.generate(
        input_ids,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

new_ids = generated_ids[0][input_ids.shape[-1]:]
answer = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
print("Prompt:", PROMPT)
print("Generated text:\n", answer)
print("주의: 생성 결과는 사실 확인이 필요할 수 있습니다.")


## Exercise

**Exercise 01 — 신경망 설정 비교**  
`practice/exercise_01_digits.py`는 Train 60% / Validation 20% / Test 20%로 분할하여 다섯 가지 설정을 검증 정확도로 비교하고, 최종 설정으로 Test를 한 번 평가합니다.

**Exercise 02 — 손실 함수 계산 비교**  
바로 아래 코드 셀에서 `predicted_scores`, `logit_binary`, `logits`를 바꿔 출력과 손실값의 관계를 비교해 보세요.

**Exercise 03 — 이미지 분류**  
`practice/7_3_image_recognition.py`에서 `UPLOAD_OWN_IMAGE = True`로 변경하고 두 이미지의 Top-5 결과를 비교해 보세요.

**Exercise 04 — Prompt 실험**  
`practice/exercise_04_prompt.py`에서 세 프롬프트와 토큰 제한 30·70의 생성 결과를 비교하세요.

전체 Quiz와 Exercise 지시문은 `README.md`에 있습니다.


In [ ]:
"""Exercise 02 — 예측값의 변화가 손실에 미치는 영향 분석.

아래 예측값을 변경한 뒤 MSE, BCE, Cross-Entropy의 변화를 관찰하세요.
이 파일은 모델을 학습하지 않고 손실 함수의 계산만 비교합니다.
"""
import torch
from torch import nn

# 회귀: 정답은 80, 90점. 아래 예측값을 바꾸어 보세요.
actual_scores = torch.tensor([80.0, 90.0])
predicted_scores = torch.tensor([78.0, 91.0])  # 변경할 값
mse = nn.MSELoss()(predicted_scores, actual_scores)

# 이진 분류: 정답이 1. 예측 Logit을 양수와 음수로 변경해 보세요.
label_binary = torch.tensor([1.0])
logit_binary = torch.tensor([2.0])            # 변경할 값
binary_probability = torch.sigmoid(logit_binary)
bce = nn.BCEWithLogitsLoss()(logit_binary, label_binary)

# 다중 클래스: 클래스 0, 1, 2 중 정답은 2. 아래 점수의 순서를 바꿔 보세요.
label_class = torch.tensor([2])
logits = torch.tensor([[1.0, 2.0, 3.0]])     # 변경할 값
class_probabilities = torch.softmax(logits, dim=1)
ce = nn.CrossEntropyLoss()(logits, label_class)

print(f"MSE: {mse.item():.4f}")
print(f"Sigmoid 출력(정답=1): {binary_probability.item():.4f} / BCE: {bce.item():.4f}")
print("Softmax 출력:", [round(float(v), 4) for v in class_probabilities[0]])
print(f"Cross-Entropy: {ce.item():.4f}")
print("예측값의 변화에 따라 각 손실이 증가하거나 감소하는지 확인하세요.")
